New ML test pipeline script


Import all libraries needed for this ML script.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
print('Imports Successfully!')

Imports Successfully!


In [3]:
#to check the python version and the path of the python executable
import sys
print(sys.executable)

/Users/boracomert/Desktop/Osint_project/.venv/bin/python


the next step: Load data 

In [4]:
DATA_PATH = "../data/processed/labeled_merged_data_cleaned.csv"

df = pd.read_csv(DATA_PATH)
print('Data Loaded Successfully!')

#the model shouldnt see layoff information becuse it can cause data leakage, so we will drop the layoff columns from the features
META_COLS = ['company', 'date', 'quarter', 'layoff','same_quarter', 'next_quarter', 'layoff_same_quarter', 'layoff_next_quarter', 'layoff_same_or_next_quarter']

FEATURE_COLS = [col for col in df.columns if col not in META_COLS]


n_companies  = df['company'].nunique()
n_rows_pos   = len(df)

print(f'Positive companies : {n_companies}')
print(f'Positive rows      : {n_rows_pos}')
print(f'Feature columns    : {len(FEATURE_COLS)}')
print(f'Quarters present   : {sorted(df["quarter"].unique())}')
df.head(5)

Data Loaded Successfully!
Positive companies : 1941
Positive rows      : 12047
Feature columns    : 346
Quarters present   : ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']


,company,date,quarter,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,...,fin_Other Non Interest Expense,fin_Depletion Income Statement,fin_Policyholder Benefits Ceded,fin_Net Income Extraordinary,same_quarter,next_quarter,layoff_same_quarter,layoff_next_quarter,layoff_same_or_next_quarter,layoff
0,AFCONS.BO,2024-09-30,2024Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q3,2024Q4,0,0,0,0
1,AFCONS.BO,2024-12-31,2024Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q4,2025Q1,0,0,0,0
2,AFCONS.BO,2025-03-31,2025Q1,367784631.0,367784631.0,1.795550e+10,2.343300e+10,5.259830e+10,7.496240e+10,3.021220e+10,...,NaN,NaN,NaN,NaN,2025Q1,2025Q2,0,0,0,0
3,AFCONS.BO,2025-06-30,2025Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2025Q2,2025Q3,0,0,0,0
4,AFCONS.BO,2025-09-30,2025Q3,367784631.0,367784631.0,3.097260e+10,3.565230e+10,5.388340e+10,8.861110e+10,3.037610e+10,...,NaN,NaN,NaN,NaN,2025Q3,2025Q4,0,0,0,0


In [5]:

# Now we will check the correlation between the feature columns, feature columns with high multicolinearity can cause issues for some models,such as KNN 
corr_matrix = df[FEATURE_COLS].corr()
corr_matrix


,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,bs_Net Tangible Assets,bs_Capital Lease Obligations,bs_Common Stock Equity,...,fin_Net Income From Tax Loss Carryforward,fin_Rent And Landing Fees,fin_Excise Taxes,fin_Securities Amortization,fin_Occupancy And Equipment,fin_Professional Expense And Contract Services Expense,fin_Other Non Interest Expense,fin_Depletion Income Statement,fin_Policyholder Benefits Ceded,fin_Net Income Extraordinary
bs_Ordinary Shares Number,1.000000,0.999833,0.563725,0.175143,0.088440,0.106398,0.120807,0.088417,0.055453,0.089080,...,-0.663042,0.190191,0.820113,-0.207808,0.871578,0.722617,0.058935,0.986963,0.179101,NaN
bs_Share Issued,0.999833,1.000000,0.564441,0.168477,0.088441,0.105381,0.121882,0.088416,0.053459,0.088871,...,-0.674448,0.184300,0.665551,-0.207803,0.884579,0.731750,0.067492,0.966393,0.013086,NaN
bs_Net Debt,0.563725,0.564441,1.000000,0.997912,0.684656,0.991991,-0.975023,0.684528,0.653954,0.949582,...,-0.706239,0.071350,0.378853,0.902989,0.864269,0.720518,0.027146,0.984310,-0.494063,0.698282
bs_Total Debt,0.175143,0.168477,0.997912,1.000000,0.372486,0.507912,0.269541,0.372479,0.482158,0.406292,...,-0.747338,0.186755,0.564773,0.921106,0.882041,0.737080,0.063920,0.985419,-0.152494,NaN
bs_Tangible Book Value,0.088440,0.088441,0.684656,0.372486,1.000000,0.988482,0.986641,1.000000,0.941486,0.999194,...,0.811359,0.133221,-0.206442,0.931169,0.891222,0.792443,0.032623,-0.986756,0.300489,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fin_Professional Expense And Contract Services Expense,0.722617,0.731750,0.720518,0.737080,0.792443,0.756972,NaN,0.794620,0.753888,0.798042,...,NaN,0.660055,NaN,0.517925,0.839100,1.000000,-0.276320,NaN,NaN,NaN
fin_Other Non Interest Expense,0.058935,0.067492,0.027146,0.063920,0.032623,0.053736,NaN,0.035496,0.139436,0.036381,...,NaN,-0.532344,NaN,0.428557,-0.132422,-0.276320,1.000000,NaN,NaN,NaN
fin_Depletion Income Statement,0.986963,0.966393,0.984310,0.985419,-0.986756,0.986247,-0.718708,-0.986756,0.995968,0.846622,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
fin_Policyholder Benefits Ceded,0.179101,0.013086,-0.494063,-0.152494,0.300489,0.254465,NaN,0.300489,-0.085225,0.280333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN


In [ ]:
#Now to train the models...

#target column is  layoff 

target_col = 'layoff'

feature_df = df[FEATURE_COLS].copy()

presence_df = df[FEATURE_COLS].notna().astype(int)  # 1 if present, 0 if NaN
presence_df.columns = [f'HAS_{c}' for c in FEATURE_COLS]


var = presence_df.var()
presence_df = presence_df.loc[:, var > 0]
print(f"Presence features after removing zero variance: {presence_df.shape[1]} columns  ")


x = pd.concat([feature_df, presence_df], axis=1)
y = df[target_col]


X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

#to prevent data leakage we do filling with mean of test data after we do our trtain/test split we will also drop columns with all the same values
nan_columns = X_train.columns[X_train.isna().all()]
if len(nan_columns) > 0:
    print("Dropping columns with all NaN values...")
    X_train = X_train.drop(columns=nan_columns)
    X_test = X_test.drop(columns=nan_columns)

train_means = X_train.mean()

X_train = X_train.fillna(train_means)
X_test = X_test.fillna(train_means)

train_var = X_train.var()
non_constant_cols = train_var[train_var > 0].index

X_train = X_train[non_constant_cols]
X_test = X_test[non_constant_cols]

print(f"Final train shape: {X_train.shape}")
print(f"Final test shape : {X_test.shape}")
print("Train label balance:")
print(y_train.value_counts())


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = [
    ('Logistic Regression', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'), True),
    ('Random Forest', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'), False),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=RANDOM_STATE,), False),
    ('XGBoost', XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'), False),
    ('SVM', SVC(random_state=RANDOM_STATE, probability=True,class_weight='balanced'), True),
    ('KNN', KNeighborsClassifier(), True),
    ('decision_tree', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'), False)
]

results = []




for model_name, model, needs_scaling in models:
    print(f'Training {model_name}...')
    
    if needs_scaling:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    report = classification_report(y_test, y_pred, output_dict=True,zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    cv_auc = cross_val_score(model, X_train_scaled if needs_scaling else X_train, y_train, cv=cv, scoring='roc_auc').mean()

    results.append({
        'model': model_name,
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1-score': report['1']['f1-score'],
        'roc_auc': roc_auc,
        'cv_roc_auc': cv_auc
    })
    
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='roc_auc', ascending=False)
print(results_df.to_string(index=False))

Presence features after removing zero variance: 346 columns  
Final train shape: (9637, 689)
Final test shape : (2410, 689)
Train label balance:
layoff
0    8673
1     964
Name: count, dtype: int64
Training Logistic Regression...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...
Training SVM...
Training KNN...
Training decision_tree...
              model  precision   recall  f1-score  roc_auc  cv_roc_auc
      Random Forest   0.369565 0.211618  0.269129 0.780615    0.773103
            XGBoost   0.434783 0.082988  0.139373 0.744234    0.752435
  Gradient Boosting   0.555556 0.041494  0.077220 0.704900    0.708495
                SVM   0.190294 0.618257  0.291016 0.700420    0.702798
                KNN   0.506667 0.157676  0.240506 0.688070    0.692877
Logistic Regression   0.160550 0.580913  0.251572 0.658949    0.646816
      decision_tree   0.273063 0.307054  0.289062 0.608397    0.597393


In [ ]:
#visualisation etc